### Dataset: https://www.kaggle.com/datasets/saisirishan/indian-vehicle-dataset/data

In [1]:
import os

%pwd

'c:\\Users\\Hp\\Documents\\GitHub\\traffic_scene\\notebooks'

In [2]:
os.chdir("../")

%pwd

'c:\\Users\\Hp\\Documents\\GitHub\\traffic_scene'

In [3]:
import shutil
from pathlib import Path
import xml.etree.ElementTree as ET


# configs
BASE_DIR = Path("dataset/anpr")
DATASET_DIR = (BASE_DIR / "cleaned")
YOLO_DIR = (BASE_DIR / "processed")
YAML_PATH = ("data.yaml")
PRO_FOLDERS = ["train", "valid", "test"]
PRO_IMG_EXTENSIONS = {".jpg", ".jpeg", ".png"}
ANNOTATION_EXT = ".xml"
YOLO_ANNOTATION_EXT = ".txt"

In [4]:
def create_yolo_dirs(pro_folders=PRO_FOLDERS, output_dir=YOLO_DIR):
    for split in pro_folders:
        (output_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (output_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

def convert_xml_to_yolo(xml_file, img_path, label_path):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        img_w = int(root.find("size/width").text)
        img_h = int(root.find("size/height").text)
        
        yolo_labels = []
        for obj in root.findall("object"):
            bbox = obj.find("bndbox")
            if bbox is None:
                print(f"[WARN] No bounding box in {xml_file}")
                continue
            
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            
            # convert to the format that yolo accepts
            x_center = (((xmin + xmax) / 2) / img_w)
            y_center = (((ymin + ymax) / 2) / img_h)
            w = ((xmax - xmin) / img_w)
            h = ((ymax - ymin) / img_h)
            
            yolo_labels.append(f"0 {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")
        
        if not yolo_labels:
            print(f"[WARN] No objects found in {xml_file}")

        # write to label file
        with open(label_path, "w") as f:
            f.write("\n".join(yolo_labels))
    except Exception as e:
        print(f"[ERROR] Failed to convert {xml_file}: {e}")


def preprocess_dataset(pro_folders=PRO_FOLDERS, dataset_dir=DATASET_DIR, output_dir=YOLO_DIR, pro_img_extensions=PRO_IMG_EXTENSIONS, annotation_ext=ANNOTATION_EXT, yolo_annotation_ext=YOLO_ANNOTATION_EXT):
    
    print("[INFO] Starting pre-processing...")
    create_yolo_dirs()
    print("[INFO] Created necessary directories for YOLO format.")
    
    for split in pro_folders:
        img_dir = dataset_dir / split
        for img_file in img_dir.iterdir():
            if img_file.suffix.lower() not in pro_img_extensions:
                continue

            sml_file = img_file.with_suffix(annotation_ext)
            target_img_path = output_dir / "images" / split / img_file.name
            target_lbl_path = output_dir / "labels" / split / (img_file.stem + yolo_annotation_ext)

            shutil.copy(img_file, target_img_path)
            if sml_file.exists():
                convert_xml_to_yolo(sml_file, img_file, target_lbl_path)
            else:
                print(f"[WARN] No annotation found for {img_file.name}, creating empty label.")
                open(target_lbl_path, "w").close()

    print("[INFO] Pre-processing completed!")


preprocess_dataset()

[INFO] Starting pre-processing...
[INFO] Created necessary directories for YOLO format.
[INFO] Pre-processing completed!


In [5]:
def validate_yolo_dataset(pro_folders, output_dir):
    """
    Validate the YOLO dataset structure:
    - Ensures images and labels are paired correctly
    - Counts number of images and annotations per split
    """
    print("[INFO] Starting YOLO dataset validation...")
    
    for split in pro_folders:
        img_dir = output_dir / "images" / split
        lbl_dir = output_dir / "labels" / split
        
        if not img_dir.exists() or not lbl_dir.exists():
            print(f"[ERROR] Missing directory for split: {split}")
            continue

        images = [f for f in img_dir.iterdir() if f.is_file()]
        labels = [f for f in lbl_dir.iterdir() if f.is_file()]
        
        image_names = {f.stem for f in images}
        label_names = {f.stem for f in labels}
        
        missing_labels = image_names - label_names
        extra_labels = label_names - image_names
        
        print(f"\n[INFO] Split: {split}")
        print(f"  Total Images: {len(images)}")
        print(f"  Total Labels: {len(labels)}")

        if missing_labels:
            print(f"  [WARN] Missing annotations for {len(missing_labels)} images:")
            for name in sorted(missing_labels):
                print(f"    - {name}")

        if extra_labels:
            print(f"  [WARN] Labels without matching images ({len(extra_labels)}):")
            for name in sorted(extra_labels):
                print(f"    - {name}")

    print("\n[INFO] YOLO dataset validation complete.")


validate_yolo_dataset(PRO_FOLDERS, YOLO_DIR)

[INFO] Starting YOLO dataset validation...

[INFO] Split: train
  Total Images: 1358
  Total Labels: 1357

[INFO] Split: valid
  Total Images: 169
  Total Labels: 169

[INFO] Split: test
  Total Images: 171
  Total Labels: 171

[INFO] YOLO dataset validation complete.


In [6]:
def find_missing_labels(output_dir, split="train"):
    img_dir = output_dir / "images" / split
    lbl_dir = output_dir / "labels" / split
    
    if not img_dir.exists() or not lbl_dir.exists():
        print(f"[ERROR] Directory not found for split: {split}")
        return

    images = [f.stem for f in img_dir.iterdir() if f.is_file()]
    labels = [f.stem for f in lbl_dir.iterdir() if f.is_file()]
    
    missing = set(images) - set(labels)
    if missing:
        print(f"[DEBUG] Missing label(s) for {len(missing)} file(s): {missing}")
    else:
        print("[DEBUG] No missing labels.")


find_missing_labels(YOLO_DIR)

[DEBUG] No missing labels.


In [7]:
def auto_fix_missing_labels(output_dir, split="train"):
    img_dir = output_dir / "images" / split
    lbl_dir = output_dir / "labels" / split
    lbl_dir.mkdir(parents=True, exist_ok=True)

    images = [f.stem for f in img_dir.iterdir() if f.is_file()]
    labels = {f.stem for f in lbl_dir.iterdir() if f.is_file()}

    for img_name in images:
        if img_name not in labels:
            empty_label = lbl_dir / (img_name + ".txt")
            open(empty_label, "w").close()
            print(f"[FIXED] Created empty label for {img_name}.txt")


auto_fix_missing_labels(YOLO_DIR)

In [8]:
import yaml

def generate_data_yaml(output_dir, yaml_path=YAML_PATH):
    """
    Generate YOLOv8 data.yaml for training.
    """
    data_config = {
        "train": str((output_dir / "images" / "train").resolve()),
        "val": str((output_dir / "images" / "valid").resolve()),
        "test": str((output_dir / "images" / "test").resolve()),
        "nc": 1,  # number of classes
        "names": ["license_plate"]  # class names
    }
    
    yaml_path = Path(output_dir) / yaml_path
    with open(yaml_path, "w") as f:
        yaml.dump(data_config, f, default_flow_style=False)
    
    print(f"[INFO] data.yaml created at: {yaml_path}")
    return yaml_path


generate_data_yaml(YOLO_DIR)

[INFO] data.yaml created at: dataset\anpr\processed\data.yaml


WindowsPath('dataset/anpr/processed/data.yaml')